<a href="https://colab.research.google.com/github/RohitGanesh7/RagSystem/blob/main/LocalModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00


In [4]:
! pip install python-docx



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.9 MB/s eta 0:00:00


In [2]:
from docx import Document

def extract_text_from_docx(path):
    doc = Document(path)
    text = ""
    for para in doc.paragraphs:
        text += para.text + "\n"
    return text

text = extract_text_from_docx("/content/Thota_Ganesh_Resume.docx")

print(text)



PROFESSIONAL SUMMARY
Software Engineer with 3.6+ years of experience in backend development, API design, and full-stack capabilities. Strong expertise in Python (FastAPI, Flask, Django REST Framework), PostgreSQL, and building scalable backend systems. Comfortable working with JavaScript, Node.js, and Next.js through real project experience. Hands-on experience integrating AI models using OpenAI and HuggingFace, and building ETL pipelines using Python & Pandas. Worked across HRTech, LegalTech, and data engineering platforms, building scalable backend systems and APIs. Strong understanding of REST APIs, microservices, data processing, cloud storage (AWS S3).
TECHNICAL SKILLS

WORK EXPERIENCE
Software Engineer   |   TechOptima Pvt Ltd, Hyderabad   |   June 2021 – Present (3.6+ Years)
Projects
Optima Management Hub — HR & Operations Management Platform   |   Jan 2025 – Present
React.js · FastAPI · PostgreSQL
A centralized HR and employee management platform designed to automate internal 

In [3]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = chunk_text(text)
print("Total chunks:", len(chunks))


Total chunks: 10


In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(chunks)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))




In [6]:
from huggingface_hub import login
login("use_token")


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Configure BitsAndBytesConfig for 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [1]:
import re

def chat_with_pdf(question, k=5):
    global chat_history

    # 1️⃣ Embed question
    question_embedding = embedding_model.encode([question])

    # 2️⃣ Retrieve relevant chunks
    D, I = index.search(np.array(question_embedding), k)

    context = ""
    for idx in I[0]:
        context += chunks[idx] + "\n"

    # 3️⃣ Prepare conversation memory (only last 3 conversations)
    history_text = ""
    for item in chat_history[-3:]:
        history_text += f"User: {item['user']}\nAssistant: {item['assistant']}\n"

    # 4️⃣ Final simplified prompt
    prompt = f"""
You are a helpful assistant.
Answer only based on the PDF context.
If the answer is not in the document, respond with "Not found in document."

Conversation History:
{history_text}

PDF Context:
{context}

Current Question:
{question}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract assistant's answer from the response
    start_marker = "Answer:"
    if start_marker in response:
        answer_start_index = response.rfind(start_marker) + len(start_marker)
        clean_response = response[answer_start_index:].strip()
    else:
        clean_response = response.strip()

    # Define regex patterns for unwanted instruction echoes and phrases
    instruction_patterns = [
        r'\(Answer only based on the PDF context\)',
        r'\(If the answer is not in the document, respond with "Not found in document\." ONLY\)',
        r'\(Do not include conversational filler, greetings, or follow-up questions\)',
        r'\(Provide only the direct answer\)',
    ]

    # Cleanup instructions from response
    min_idx = len(clean_response)
    for pattern in instruction_patterns:
        match = re.search(pattern, clean_response, re.IGNORECASE)
        if match:
            min_idx = min(min_idx, match.start())

    clean_response = clean_response[:min_idx].strip()

    # General cleanup for unwanted filler words
    unwanted_phrases = [
        "Assistant:", "User:", "Bot:", "Please go ahead and ask your next question.",
        "I'm here to help!", "Waiting for your next question...",
        "Can you clarify?", "What else would you like to know?"
    ]
    for phrase in unwanted_phrases:
        clean_response = clean_response.replace(phrase, "").strip()

    # Clean up repeated "Not found in document."
    clean_response = re.sub(r'(?i)(Not found in document\.)(\s*\1)+', r'\1', clean_response).strip()
    clean_response = re.sub(r'\s*\n\s*', '\n', clean_response).strip()
    clean_response = re.sub(r'\s+', ' ', clean_response).strip()  # Consolidate multiple spaces

    # 5️⃣ Save conversation
    chat_history.append({
        "user": question,
        "assistant": clean_response
    })

    return clean_response

In [ ]:
while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    answer = chat_with_pdf(user_input)
    print("\nBot:", answer)


You: hi who are you?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Bot: I am a helpful assistant. I am here to assist you based on the PDF context. I can answer any questions you have about the provided information. If the answer is not found in the document, I will respond with "Not found in document". What would you like to know?
You: who is this resume?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Bot: This is a resume of a software engineer with 3.6+ years of experience in backend development, API design, and full-stack capabilities. The resume highlights the candidate's expertise in Python, PostgreSQL, and building scalable backend systems, among other technical skills. The candidate has worked on various projects, including the Optima Management Hub, a centralized HR and employee management platform.


In [9]:
chat_history = []